# Within-Dataset Benchmark: Ridge, Random Forest, LightGBM, GraphDRP, SimpleLinearNN

This notebook runs the first benchmark stage using the official IMPROVE within-dataset splits. It starts with five models:

- `ridge`
- `random_forest`
- `lightgbm`
- `graphdrp`
- `simple_linear_nn`

The code is kept in `within_dataset_4models.py` so the model registry can be extended later without turning the notebook into a maze.

In [1]:
from pathlib import Path
import importlib
import sys

ROOT = Path.cwd()
if ROOT.name == "new_notebook":
    ROOT = ROOT.parent

NOTEBOOK_DIR = ROOT / "new_notebook"
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

import within_dataset_4models
importlib.reload(within_dataset_4models)
from within_dataset_4models import make_default_config, run_within_dataset_benchmark, summarize_results

cfg = make_default_config(ROOT)
cfg

BenchmarkConfig(root=PosixPath('/Users/vietanh/Desktop/ML-Predicting-drug-response'), datasets=['CCLE'], folds=[0], models=['ridge', 'random_forest', 'lightgbm', 'graphdrp', 'simple_linear_nn'], use_lincs_symbol_genes=True, top_ge_features=512, top_mordred_features=512, max_train_rows=None, max_eval_rows=None, random_forest_epochs=100, random_forest_patience=50, graphdrp_epochs=150, graphdrp_batch_size=256, graphdrp_patience=20, graphdrp_learning_rate=0.0001, simple_nn_epochs=300, simple_nn_batch_size=64, simple_nn_val_batch_size=64, simple_nn_patience=50, simple_nn_learning_rate=0.01, simple_nn_dropout=0.01, simple_nn_model='default', random_state=42)

## Config

Default is a quick smoke test on `CCLE`, fold `0`. For the full paper-style within-dataset run, switch to all five datasets and ten folds.

In [2]:
# Smoke test
cfg.datasets = ["CCLE"]
cfg.folds = [0]

# Full within-dataset benchmark
# cfg.datasets = ["CCLE", "CTRPv2", "gCSI", "GDSCv1", "GDSCv2"]
# cfg.folds = list(range(10))

cfg.models = ["ridge", "random_forest", "lightgbm", "graphdrp", "simple_linear_nn"]
cfg.top_ge_features = 512
cfg.top_mordred_features = 512
cfg.graphdrp_epochs = 150
cfg.graphdrp_patience = 20
cfg.simple_nn_epochs = 300
cfg.simple_nn_patience = 50
cfg.simple_nn_model = "default"

# Use these for a very quick debug run, then set back to None.
cfg.max_train_rows = None
cfg.max_eval_rows = None

cfg

BenchmarkConfig(root=PosixPath('/Users/vietanh/Desktop/ML-Predicting-drug-response'), datasets=['CCLE'], folds=[0], models=['ridge', 'random_forest', 'lightgbm', 'graphdrp', 'simple_linear_nn'], use_lincs_symbol_genes=True, top_ge_features=512, top_mordred_features=512, max_train_rows=None, max_eval_rows=None, random_forest_epochs=100, random_forest_patience=50, graphdrp_epochs=150, graphdrp_batch_size=256, graphdrp_patience=20, graphdrp_learning_rate=0.0001, simple_nn_epochs=300, simple_nn_batch_size=64, simple_nn_val_batch_size=64, simple_nn_patience=50, simple_nn_learning_rate=0.01, simple_nn_dropout=0.01, simple_nn_model='default', random_state=42)

## Run Benchmark

GraphDRP requires `torch`, `torch-geometric`, and `rdkit`. SimpleLinearNN requires `torch`. If dependencies are missing, the runner records the model as skipped instead of stopping the whole benchmark.

In [3]:
results = run_within_dataset_benchmark(cfg)
display(results)

Dataset=CCLE | fold=0
Rows usable: train=7,616, val=952, test=951 | tabular features=1,024
  Training ridge
    val: n=952 RMSE=0.0821 MAE=0.0630 R2=0.7174 Pearson=0.8473
    test: n=951 RMSE=0.0843 MAE=0.0655 R2=0.7388 Pearson=0.8596
  Training random_forest
    val: n=952 RMSE=0.0796 MAE=0.0621 R2=0.7346 Pearson=0.8572
    test: n=951 RMSE=0.0835 MAE=0.0648 R2=0.7437 Pearson=0.8625
  Training lightgbm


/opt/anaconda3/envs/demo/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/demo/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


    val: n=952 RMSE=0.0694 MAE=0.0543 R2=0.7979 Pearson=0.8936
    test: n=951 RMSE=0.0719 MAE=0.0558 R2=0.8099 Pearson=0.9005
  Training graphdrp


/Users/vietanh/Desktop/ML-Predicting-drug-response/new_notebook/within_dataset_4models.py:771: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:256.)
  x = torch.tensor([atom_features(atom) for atom in mol.GetAtoms()], dtype=torch.float)


    val: n=952 RMSE=0.0745 MAE=0.0579 R2=0.7670 Pearson=0.8850
    test: n=951 RMSE=0.0768 MAE=0.0603 R2=0.7834 Pearson=0.8900
  Training simple_linear_nn
    val: n=952 RMSE=0.0766 MAE=0.0602 R2=0.7541 Pearson=0.8696
    test: n=951 RMSE=0.0782 MAE=0.0606 R2=0.7756 Pearson=0.8813
Saved: /Users/vietanh/Desktop/ML-Predicting-drug-response/new_notebook/results/within_dataset_4models_results.csv


,analysis,dataset,fold,stage,model,train_seconds,n_train,n_features,status,n,rmse,mae,r2,pearson,preprocess,split_train_rows,split_val_rows,split_test_rows
0,within_dataset,CCLE,0,val,ridge,0.513860,7616,1024,ok,952,0.082099,0.062970,0.717354,0.847328,tabular gene expression + Mordred drug descrip...,7616,952,951
1,within_dataset,CCLE,0,test,ridge,0.513860,7616,1024,ok,951,0.084335,0.065521,0.738774,0.859590,tabular gene expression + Mordred drug descrip...,7616,952,951
2,within_dataset,CCLE,0,val,random_forest,165.800348,7616,1024,ok,952,0.079557,0.062060,0.734584,0.857179,official-style tabular gene expression + Mordr...,7616,952,951
3,within_dataset,CCLE,0,test,random_forest,165.800348,7616,1024,ok,951,0.083534,0.064802,0.743710,0.862472,official-style tabular gene expression + Mordr...,7616,952,951
4,within_dataset,CCLE,0,val,lightgbm,11.756680,7616,1024,ok,952,0.069424,0.054349,0.797889,0.893590,tabular gene expression + Mordred drug descrip...,7616,952,951
5,within_dataset,CCLE,0,test,lightgbm,11.756680,7616,1024,ok,951,0.071936,0.055816,0.809935,0.900544,tabular gene expression + Mordred drug descrip...,7616,952,951
6,within_dataset,CCLE,0,val,graphdrp,1061.379126,7616,512,ok,952,0.074533,0.057869,0.767046,0.885010,official-style drug molecular graph from SMILE...,7616,952,951
7,within_dataset,CCLE,0,test,graphdrp,1061.379126,7616,512,ok,951,0.076787,0.060315,0.783440,0.889977,official-style drug molecular graph from SMILE...,7616,952,951
8,within_dataset,CCLE,0,val,simple_linear_nn,33.864579,7616,1024,ok,952,0.076575,0.060183,0.754108,0.869559,official-style PyTorch MLP on gene expression ...,7616,952,951
9,within_dataset,CCLE,0,test,simple_linear_nn,33.864579,7616,1024,ok,951,0.078158,0.060612,0.775636,0.881335,official-style PyTorch MLP on gene expression ...,7616,952,951


## Summary

In [4]:
summary = summarize_results(results, cfg.out_dir)
display(summary)

Saved: /Users/vietanh/Desktop/ML-Predicting-drug-response/new_notebook/results/within_dataset_4models_summary.csv


,dataset,model,folds,n_test_mean,r2_mean,r2_std,rmse_mean,rmse_std,mae_mean,mae_std,pearson_mean,pearson_std,train_seconds_mean
1,CCLE,lightgbm,1,951.0,0.809935,NaN,0.071936,NaN,0.055816,NaN,0.900544,NaN,11.756680
0,CCLE,graphdrp,1,951.0,0.783440,NaN,0.076787,NaN,0.060315,NaN,0.889977,NaN,1061.379126
4,CCLE,simple_linear_nn,1,951.0,0.775636,NaN,0.078158,NaN,0.060612,NaN,0.881335,NaN,33.864579
2,CCLE,random_forest,1,951.0,0.743710,NaN,0.083534,NaN,0.064802,NaN,0.862472,NaN,165.800348
3,CCLE,ridge,1,951.0,0.738774,NaN,0.084335,NaN,0.065521,NaN,0.859590,NaN,0.513860
